# Train YOLOv8n-seg on synthetic scratches (synth_v1)

Before running: on the Mac, zip the dataset and upload it to Google Drive as `MyDrive/sdf/synth_v1.zip`:
```
cd data/yolo && zip -qr synth_v1.zip synth_v1
```
Runtime → Change runtime type → **T4 GPU**. Results (weights, plots, MLflow logs) are saved to `MyDrive/sdf/`, so they survive a disconnect.

Steps: 1) mount Drive, 2) install, 3) unpack data, 4) 1-epoch smoke test, 5) full training.

In [ ]:
# 1) Google Drive: input zip + output folder
from google.colab import drive
drive.mount('/content/drive')
SDF = '/content/drive/MyDrive/sdf'

In [ ]:
# 2) Install (pinned versions, so results are reproducible)
!pip -q install ultralytics==8.3.40 mlflow==2.17.2
import os
from ultralytics import YOLO, settings
# experiment tracking: every run logs its params + metrics to Drive
os.environ['MLFLOW_TRACKING_URI'] = f'file:{SDF}/mlruns'
os.environ['MLFLOW_EXPERIMENT_NAME'] = 'sim2real_scratch'
settings.update({'mlflow': True})

In [ ]:
# 3) Unpack the dataset to the fast local disk (reading from Drive during training is slow)
!mkdir -p /content/yolo && unzip -qo {SDF}/synth_v1.zip -d /content/yolo
!ls /content/yolo/synth_v1/images/train | wc -l   # expect 4500
!ls /content/yolo/synth_v1/images/val | wc -l     # expect 500
open('/content/synth_v1.yaml', 'w').write(
    'path: /content/yolo/synth_v1\ntrain: images/train\nval: images/val\nnames:\n  0: scratch\n')

In [ ]:
# 4) Smoke test: 1 epoch on 5% of the data. Only checks that everything runs.
YOLO('yolov8n-seg.pt').train(data='/content/synth_v1.yaml', epochs=1, imgsz=512,
                             fraction=0.05, project=f'{SDF}/runs', name='smoke', exist_ok=True)

In [ ]:
# 5) Full training: baseline = synthetic only (synth_v1). ~1 h on a T4.
#    patience: stop early if val metrics do not improve for 15 epochs.
model = YOLO('yolov8n-seg.pt')  # COCO-pretrained (transfer learning)
model.train(data='/content/synth_v1.yaml', epochs=60, imgsz=512, batch=32, seed=0,
            patience=15, project=f'{SDF}/runs', name='synth_v1_baseline', exist_ok=True)
# best weights: MyDrive/sdf/runs/synth_v1_baseline/weights/best.pt